In [ ]:
import os
import sys
from pathlib import Path
import logging

# Ensure the project root and src/ are on PYTHONPATH.
project_root = Path.cwd().resolve()
if project_root.name == "src":
    project_root = project_root.parent
src_dir = project_root / "src"
for path_entry in (project_root, src_dir):
    if str(path_entry) not in sys.path:
        sys.path.insert(0, str(path_entry))

# Reinicie o kernel após alterar a configuração de dispositivo.
USAR_CPU = True
os.environ["KERAS_BACKEND"] = "tensorflow"
if USAR_CPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import keras
import pandas as pd
from utils import train_and_forecast

if keras.backend.backend() != "tensorflow":
    raise RuntimeError("Reinicie o kernel para iniciar o Keras com TensorFlow.")
print(f"Keras: {keras.backend.backend()} | dispositivo: {keras.ops.convert_to_tensor(0).device}")

# Reduce repeated utility logs during notebook training.
for logger_name in ("utils.data_utils", "src.utils.data_utils"):
    logging.getLogger(logger_name).setLevel(logging.WARNING)

# ============================================================================
# GLOBAL SETTINGS
# ============================================================================

# Available models: 'RNN', 'LSTM', or 'CNN'
MODEL_NAME = "RNN"

# Forecast horizon: 3 or 12 months
FORECAST_HORIZON = 3

# Fixed training parameters
LOOK_BACK = 12
EPOCHS = 150
BATCH_SIZE = 1
SAVE_ONNX = True
SILENCE_TRAINING = True

# Regions and product series
REGIONS = ["ilheus", "itabuna"]
PRODUCTS = [
    "acucar",
    "arroz",
    "banana",
    "cafe",
    "carne",
    "farinha",
    "feijao",
    "leite",
    "manteiga",
    "oleo",
    "pao",
    "tomate",
]


# Previsão preço da cesta básica

## Treinamento de modelos

### Cesta Básica

In [ ]:
# Basic basket forecast

print(f"Training basic basket: {MODEL_NAME}, {FORECAST_HORIZON}-month horizon")

basket_summary = []
for region in REGIONS:
    basket_summary.append(
        train_and_forecast(
            region=region,
            series_name="cesta_basica",
            forecast_type="cesta",
            model_name=MODEL_NAME,
            look_back=LOOK_BACK,
            forecast_horizon=FORECAST_HORIZON,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            save_onnx=SAVE_ONNX,
            silence_training=SILENCE_TRAINING,
        )
    )

pd.DataFrame(basket_summary)


### Produtos Individuais

In [ ]:
# Product forecasts

total_models = len(REGIONS) * len(PRODUCTS)
print(
    f"Training products: {MODEL_NAME}, {FORECAST_HORIZON}-month horizon "
    f"({total_models} models)"
)

product_summary = []
for region in REGIONS:
    print(f"{region.upper()}: training {len(PRODUCTS)} products...")
    for product in PRODUCTS:
        product_summary.append(
            train_and_forecast(
                region=region,
                series_name=product,
                forecast_type="produtos",
                model_name=MODEL_NAME,
                look_back=LOOK_BACK,
                forecast_horizon=FORECAST_HORIZON,
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                save_onnx=SAVE_ONNX,
                silence_training=SILENCE_TRAINING,
                subdir="produtos",
            )
        )
    print(f"{region.upper()}: done")

pd.DataFrame(product_summary)
